In [7]:
from notebooks import nb_setup
import torch
import random
import numpy as np
import pandas as pd
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModel
from src.data import load_tokenised_data
from src.answerability.evaluation import evaluate_answerability_predictions
from src.answerability.gru_dataset import PretrainedAnswerabilityDataset, make_collate_fn
from src.answerability.logistic_regression import (
    train_logistic_regression,
    predict_logistic_regression,
    logistic_feature_weights,
)
from src.answerability.gru_model import (
    PretrainedGRUClassifier,
    train_one_epoch_answerability,
    predict_answerability,
    evaluate_macro_f1
)
from src.rule_based import rule_based_classifier, STOPWORDS
from src.translator import NLLBTranslator
translator = NLLBTranslator()

config.json:   0%|          | 0.00/808 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 4.85MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.3MB            

...

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/1016 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 5.48GB            

model.safetensors: downloading bytes:           |  0.00B            

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

In [8]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

In [9]:
df_train_tokenised, df_val_tokenised = load_tokenised_data()

In [10]:
lr_model, lr_vectorizer = train_logistic_regression(df_train_tokenised)

In [11]:
lr_pred, lr_prob = predict_logistic_regression(lr_model, lr_vectorizer, df_val_tokenised)

In [12]:
y_val = df_val_tokenised["answerable"].astype(int).to_numpy()
val_langs = df_val_tokenised["lang"].to_numpy()

In [13]:
lr_results = evaluate_answerability_predictions(y_val, lr_pred, val_langs, "Logistic Regression").round(4)
lr_results

,Model,Language,N,Macro F1,Unanswerable Precision,Unanswerable Recall,Unanswerable F1,Answerable Precision,Answerable Recall,Answerable F1
0,Logistic Regression,Overall,1155,0.7236,0.8108,0.3659,0.5042,0.9038,0.9859,0.9431
1,Logistic Regression,Arabic,415,0.9459,0.8889,0.9231,0.9057,0.9889,0.9835,0.9862
2,Logistic Regression,Korean,356,0.4863,0.0000,0.0000,0.0000,0.9466,1.0000,0.9726
3,Logistic Regression,Telugu,384,0.5383,0.6000,0.1290,0.2124,0.7775,0.9725,0.8641


In [14]:
feature_weights = logistic_feature_weights(lr_model, lr_vectorizer)

In [15]:
feature_weights.head(15)

,feature,weight
0,q_word=هل,-6.678970
1,q_word=수,-1.204230
2,q_word=있을까,-0.915575
3,q_word=있는가,-0.911634
4,c_word=center,-0.874684
5,q_word=దేశం,-0.872377
6,c_word=armed,-0.788823
7,c_word=elections,-0.755297
8,q_word=ريشتر,-0.749401
9,q_word=الجيولوجي,-0.749401


In [16]:
feature_weights.tail(15)

,feature,weight
56966,q_word=무엇인가요,0.543971
56967,c_word=official,0.553744
56968,q_word=أين,0.565577
56969,c_word=following,0.576461
56970,q_word=어디인가,0.621662
56971,c_word=so,0.623711
56972,c_word=.,0.643672
56973,q_word=ఎన్ని,0.662118
56974,q_word=كم,0.693730
56975,context_has_digit,0.738529


In [33]:
majority_class = df_train_tokenised["answerable"].astype(int).mode()[0]
majority_pred = np.full(len(y_val), majority_class)

majority_results = evaluate_answerability_predictions(y_val, majority_pred, val_langs, "Majority Baseline").round(4)
majority_results

,Model,Language,N,Macro F1,Unanswerable Precision,Unanswerable Recall,Unanswerable F1,Answerable Precision,Answerable Recall,Answerable F1
0,Majority Baseline,Overall,1155,0.4618,0.0,0.0,0.0,0.8580,1.0,0.9236
1,Majority Baseline,Arabic,415,0.4666,0.0,0.0,0.0,0.8747,1.0,0.9332
2,Majority Baseline,Korean,356,0.4863,0.0,0.0,0.0,0.9466,1.0,0.9726
3,Majority Baseline,Telugu,384,0.4311,0.0,0.0,0.0,0.7578,1.0,0.8622


In [32]:
rule_pred = df_val_tokenised.apply(
    lambda row: rule_based_classifier(row["question"], row["context"], row["lang"], translator),
    axis=1
).astype(int).to_numpy()

rule_results = evaluate_answerability_predictions(y_val, rule_pred, val_langs, "Rule-Based").round(4)
rule_results

,Model,Language,N,Macro F1,Unanswerable Precision,Unanswerable Recall,Unanswerable F1,Answerable Precision,Answerable Recall,Answerable F1
0,Rule-Based,Overall,1155,0.5683,0.2400,0.4024,0.3007,0.8886,0.7891,0.8359
1,Rule-Based,Arabic,415,0.5280,0.1748,0.3462,0.2323,0.8910,0.7658,0.8237
2,Rule-Based,Korean,356,0.4569,0.0460,0.2105,0.0755,0.9442,0.7537,0.8383
3,Rule-Based,Telugu,384,0.6709,0.5176,0.4731,0.4944,0.8361,0.8591,0.8475


In [17]:
MODEL_NAME = "xlm-roberta-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
xlmr = AutoModel.from_pretrained(MODEL_NAME)

pretrained_embeddings = (
    xlmr
    .get_input_embeddings()
    .weight
    .detach()
    .cpu()
    .clone()
)

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.12GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [18]:
train_dataset = PretrainedAnswerabilityDataset(df_train_tokenised, tokenizer)
val_dataset = PretrainedAnswerabilityDataset(df_val_tokenised, tokenizer)

collate_fn = make_collate_fn(tokenizer.pad_token_id)

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    collate_fn=collate_fn,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False,
    collate_fn=collate_fn,
)

In [19]:
model = PretrainedGRUClassifier(
    pretrained_embeddings=pretrained_embeddings,
    hidden_dim=64,
    dropout_prob=0.3,
    padding_idx=tokenizer.pad_token_id,
).to(device)

criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    filter(
        lambda p: p.requires_grad,
        model.parameters(),
    ),
    lr=1e-3,
    weight_decay=1e-4,
)

In [20]:
NUM_EPOCHS = 10
PATIENCE = 3

best_val_f1 = -1
best_model_state = None
best_epoch = None

epochs_without_improvement = 0

for epoch in range(NUM_EPOCHS):

    train_loss = (
        train_one_epoch_answerability(
            model,
            train_loader,
            optimizer,
            criterion,
            device,
        )
    )

    val_f1 = evaluate_macro_f1(model, val_loader, device)

    if val_f1 > best_val_f1:

        best_val_f1 = val_f1
        best_epoch = epoch + 1

        best_model_state = {
            key: value.detach().cpu().clone() for key, value in model.state_dict().items()
        }
        epochs_without_improvement = 0

    else:
        epochs_without_improvement += 1

    print(
        f"Epoch {epoch + 1}/{NUM_EPOCHS} | "
        f"Train loss: {train_loss:.4f} | "
        f"Val Macro F1: {val_f1:.4f}"
    )

    if epochs_without_improvement >= PATIENCE:
        print("Early stopping")
        break

Epoch 1/10 | Train loss: 0.1697 | Val Macro F1: 0.6986
Epoch 2/10 | Train loss: 0.0942 | Val Macro F1: 0.6986
Epoch 3/10 | Train loss: 0.0815 | Val Macro F1: 0.7727
Epoch 4/10 | Train loss: 0.0738 | Val Macro F1: 0.8781
Epoch 5/10 | Train loss: 0.0700 | Val Macro F1: 0.8659
Epoch 6/10 | Train loss: 0.0574 | Val Macro F1: 0.8416
Epoch 7/10 | Train loss: 0.0536 | Val Macro F1: 0.8521
Early stopping


In [31]:
print(f"Best epoch: {best_epoch}")
print(f"Best validation Macro F1: {best_val_f1:.4f}")

Best epoch: 4
Best validation Macro F1: 0.8781


In [21]:
model.load_state_dict(best_model_state)
model.to(device)

PretrainedGRUClassifier(
  (embeddings): Embedding(250002, 768, padding_idx=1)
  (gru): GRU(768, 64, batch_first=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (classifier): Linear(in_features=256, out_features=1, bias=True)
)

In [22]:
gru_pred, gru_prob, y_val, val_langs = predict_answerability(model, val_loader, device)
gru_results = evaluate_answerability_predictions(y_val, gru_pred, val_langs, "Pretrained GRU").round(4)
gru_results

,Model,Language,N,Macro F1,Unanswerable Precision,Unanswerable Recall,Unanswerable F1,Answerable Precision,Answerable Recall,Answerable F1
0,Pretrained GRU,Overall,1155,0.8781,0.8623,0.7256,0.7881,0.9558,0.9808,0.9681
1,Pretrained GRU,Arabic,415,0.9574,0.8929,0.9615,0.9259,0.9944,0.9835,0.9889
2,Pretrained GRU,Korean,356,0.8835,0.7273,0.8421,0.7805,0.9910,0.9822,0.9866
3,Pretrained GRU,Telugu,384,0.8082,0.8833,0.5699,0.6928,0.8765,0.9759,0.9236


In [23]:
errors_df = df_val_tokenised.copy()
errors_df["true"] = y_val
errors_df["pred"] = gru_pred
errors_df["error_type"] = None
errors_df.loc[(errors_df["true"] == 1) & (errors_df["pred"] == 0), "error_type"] = "false_negative"
errors_df.loc[(errors_df["true"] == 0) & (errors_df["pred"] == 1), "error_type"] = "false_positive"
errors_only = errors_df[errors_df["error_type"].notna()]

pd.crosstab(errors_only["lang"], errors_only["error_type"])

error_type,false_negative,false_positive
lang,,
ar,6,2
ko,6,3
te,7,40


In [34]:
all_results = pd.concat([majority_results, rule_results, lr_results, gru_results], ignore_index=True).round(4)
all_results

,Model,Language,N,Macro F1,Unanswerable Precision,Unanswerable Recall,Unanswerable F1,Answerable Precision,Answerable Recall,Answerable F1
0,Majority Baseline,Overall,1155,0.4618,0.0000,0.0000,0.0000,0.8580,1.0000,0.9236
1,Majority Baseline,Arabic,415,0.4666,0.0000,0.0000,0.0000,0.8747,1.0000,0.9332
2,Majority Baseline,Korean,356,0.4863,0.0000,0.0000,0.0000,0.9466,1.0000,0.9726
3,Majority Baseline,Telugu,384,0.4311,0.0000,0.0000,0.0000,0.7578,1.0000,0.8622
4,Rule-Based,Overall,1155,0.5683,0.2400,0.4024,0.3007,0.8886,0.7891,0.8359
5,Rule-Based,Arabic,415,0.5280,0.1748,0.3462,0.2323,0.8910,0.7658,0.8237
6,Rule-Based,Korean,356,0.4569,0.0460,0.2105,0.0755,0.9442,0.7537,0.8383
7,Rule-Based,Telugu,384,0.6709,0.5176,0.4731,0.4944,0.8361,0.8591,0.8475
8,Logistic Regression,Overall,1155,0.7236,0.8108,0.3659,0.5042,0.9038,0.9859,0.9431
9,Logistic Regression,Arabic,415,0.9459,0.8889,0.9231,0.9057,0.9889,0.9835,0.9862


In [43]:
sample_errors = (
    errors_only
    .groupby(["lang", "error_type"], group_keys=False)
    .head(2)
)

sample_errors[
    ["lang", "error_type", "question", "context"]
]

,lang,error_type,question,context
35,ar,false_negative,هل مصرهي دولة نامية؟,"In recent years, new terms have emerged to des..."
112,ar,false_negative,هل يعتبر بولس من حواري المسيح؟,"Paul the Apostle (; ; ; c. 5 – c. 64 or 67), c..."
191,ar,false_positive,لماذا تم تسمية نبتة القديس يوحنا المثقبة بهذا ...,"The specific epithet ""perforatum"" is Latin, re..."
341,ar,false_positive,ماكان سبب إندلاع الحرب الأهلية الأنغولية ؟,Möngke Khan's brother Hulagu Khan broke off hi...
520,ko,false_positive,2019년 현재 페터 베르크만 사건은 해결되었는가?,The Peter Bergmann case is an unsolved mystery...
529,ko,false_positive,수도가 없는 나라는 몇 개국인가?,"A few nation states have multiple capitals, an..."
530,ko,false_negative,소아마비 바이러스의 증상은?,"Poliomyelitis, often called polio or infantile..."
550,ko,false_negative,오존층은 무엇으로 만들어졌는가?,The photochemical mechanisms that give rise to...
814,te,false_negative,అత్యంత శక్తివంతమైన నావికా రక్షణ దళం ఏ దేశానికి...,The United States Navy (USN) is the naval warf...
867,te,false_negative,లోకోమోటివ్ బాయిలరును మొదట ఏ దేశంలో ప్రవేశపెట్ట...,A steam locomotive is a locomotive whose prima...


In [51]:

category_map = {
    520: "Possible annotation issue",
    529: "Related information, incomplete answer",
    530: "Relevant evidence missed",
    550: "Relevant evidence missed",
    35: "Partial or indirect evidence",
    112: "Relevant evidence missed",
    191: "Related information, incomplete answer",
    341: "Question-context mismatch",
    814: "Relevant evidence missed",
    867: "Partial or indirect evidence",
    1058: "Related information, incomplete answer",
    1059: "Question-context mismatch",
}

sample_errors = sample_errors.copy()
sample_errors["category"] = sample_errors.index.map(category_map)

print(sample_errors["category"].value_counts())
display(
    sample_errors[
        ["lang", "error_type", "category"]
    ].sort_values(["lang", "error_type"])
)


category
Relevant evidence missed                  4
Related information, incomplete answer    3
Partial or indirect evidence              2
Question-context mismatch                 2
Possible annotation issue                 1
Name: count, dtype: int64


,lang,error_type,category
35,ar,false_negative,Partial or indirect evidence
112,ar,false_negative,Relevant evidence missed
191,ar,false_positive,"Related information, incomplete answer"
341,ar,false_positive,Question-context mismatch
530,ko,false_negative,Relevant evidence missed
550,ko,false_negative,Relevant evidence missed
520,ko,false_positive,Possible annotation issue
529,ko,false_positive,"Related information, incomplete answer"
814,te,false_negative,Relevant evidence missed
867,te,false_negative,Partial or indirect evidence
